# Fine-Tune Workflow Tutorial

GenSec requires the following files:

- input structures (molecule, surface)
- parameters.json
- supporting folder with ASE calculators

What the code does:

- Gen: quasi-random structure search of molecule on surface (with commensurate supercell determination and preliminary force-checks with foundation model).
- [optional] Fine-tuning of foundation MLIP with the Gen database: the Gen database can be used in an iterative fashion to fine-tune a foundation model until desired accuracy.
- Sec: strucutre relaxation of extended structures from the Gen database (with fine-tuned model or other calculator).

All these stages can be run independently and turned on/off. The fine-tune stage has two sub-stages, labelling and training, that are also independent.

`parameters.json` is the command center for this workflow. All setup choices, file paths, format declarations, and fine-tuning options are directly or indirectly controlled there, so each tutorial step below is explained in direct relation to that file.

Knowledge of core functionality of the code is often assumed (Gen and Sec parts). We'll not delve into all the Gen and Sec phases parameters.

## Step 1. Prepare the Input Files
### i) Structures (molecule and surface)

Provide two input structures:
- molecule template 
- fixed frame / surface

Formats are not restricted to AIMS. GenSec reads the format from the JSON fields.

For this example, we use structures coming from FHI-aims geometry optimizations (AIMS format). Important: optimized structures are **recommended**. GenSec only requires readable files + declared formats.

Example snippet (from `parameters.json`):

```json
"geometry": {
  "filename": "MePTCDI.in",
  "format": "aims"
},
"fixed_frame": {
  "activate": true,
  "filename": "graphene.in",
  "format": "aims",
  "is_unit_cell": true
}
```

### ii) Set Up the `supporting/` Folder

Create a folder named `supporting` in the run directory (same location as `parameters.json`).

This folder stores ASE calculator scripts that GenSec imports at runtime.

There are three instances where a calculator is needed:
- For the force-check in the generation (Gen) step (ruling out unphysical structures)
- For structure relaxation in the search (Sec) step
- For labeling the fine-tuning datasets

The first two are controlled by the `calculator` block in `parameters.json`.
The fine-tuning labeling calculator is controlled by the `fine_tuning` block.

```json
"calculator": {
  "supporting_files_folder": "supporting",
  "ase_parameters_file": "mace_command.py"
},
"fine_tuning": {
  "supporting_files_folder": "supporting",
  "ase_parameters_file": "aims_command.py"
}
```

Examples of two calculators can be found in:

```json
supporting/mace_command.py
supporting/aims_command.py
```

For an AIMS calculator, the PATH to the AIMS binary and species directories are set up in the supporting file at this stage, together with your preferred "run" command (srun in the example below). From this file, every AIMS argument can be explicitly set by the user.

### iii) Set Up the Generation Parameters

This is beyond the scope of this fine-tuning tutorial.

Use `parameters.json` as reference for all generation-related settings (`protocol.generate`, configuration sampling, clashes, supercell options, etc.).

### iv) Choose the Model to Fine-Tune and Set MACE Parameters

This is controlled in the `fine_tuning` block in `parameters.json`.

There are many arguments that can be passed here; many have defaults and all of them are overrideable.

Relevant fields for multihead finetuning (explained):

- `activate`: Enables the fine-tuning pipeline when `true`.
- `loop_activate`: Enables iterative/looped fine-tuning rounds (label -> train -> evaluate -> extend dataset).


- `k_density`: Target reciprocal-space density used to build per-structure `k_grid` during labeling.

- `test_set_size`: Number of structures reserved as a fixed test set for loop evaluation.
- `foundation_model`: Name/path of the starting pretrained MACE model to fine-tune.

- `mace_output_name`: Base run name used for MACE outputs/checkpoints/log folders.
- `mace_args`: Dictionary of arguments forwarded to MACE training. Some arguments include:

    - `multiheads_finetuning`: Enables multi-head fine-tuning mode in MACE. All the following block of keyworkds refer to multihead specifically, not needed if "naive" fine-tuning is used:

        - `pt_train_file`: Pretraining dataset source for multi-head mode (for example `mp`).
        - `num_samples_pt`: Number of pretraining samples to draw/use.
        - `filter_type_pt`: Filtering strategy for selecting pretraining data.
        - `subselect_pt`: Subselection method for pretraining subset creation.
        - `weight_pt`: Relative loss weight for the pretraining head/task.
        - `atomic_numbers`: Chemical elements included in training (as atomic numbers).
        - `heads`: Head-specific settings for multi-head training.
            - `default`: Name of the default fine-tuning head in this example.
            - `E0s`: Per-element reference energies used by the head (keys are atomic numbers).

    - `swa`: Enables/disables stochastic weight averaging.
    - `batch_size`: Training batch size.
    - `valid_batch_size`: Validation batch size.
    - `max_num_epochs`: Maximum number of training epochs.

- `rmse_energy_target`: Loop stop criterion for energy RMSE (meV/atom style target, depending on MACE reporting units).
- `rmse_force_target`: Loop stop criterion for force RMSE (unit interpretation follows MACE outputs).

- `fps_batch_size`: Number of new optimally selected structures added per loop round.

- `state_file`: JSON file storing loop state/progress between runs.
- `global_labeled_db`: Cumulative labeled ASE DB used across rounds.

Example snippet (from `parameters.json`):

```json
"fine_tuning": {
    "activate": true,
    "loop_activate": true,

    "supporting_files_folder": "supporting",
    "ase_parameters_file": "aims_command.py",

    "k_density": 30.0,

    "test_set_size": 25,

    "foundation_model": "medium-0b3",
    "mace_output_name": "test-loop",
    "mace_args": {

        "multiheads_finetuning": true,
        "pt_train_file": "mp",
        "num_samples_pt": 400,
        "filter_type_pt": "combinations",
        "subselect_pt": "random",
        "weight_pt": 1,
        "atomic_numbers": [1, 6, 7, 8],
        "swa": false,
        "batch_size": 4,
        "valid_batch_size": 4,
        "max_num_epochs": 30,
        "heads": {
                "default": {
                        "E0s": {
                                "1": -13.5980301782,
                                "6": -1029.1067769881,
                                "7": -1485.3076478427,
                                "8": -2043.2399676739
                                }
                }
        }
    },

    "rmse_energy_target": 2.0,
    "rmse_force_target": 50.0,
    "fps_batch_size": 12,

    "state_file": "fine_tune_state.json",
    "global_labeled_db": "db_labeled_global.db"
}
```

### Add a naive fine-tuning example?

# Here is where we lose all the black-boxiness. Impossible to keep it balck-boxy with MultiHeads fine-tuning, it is very slow and on top of that there are a million choices to make. We need to discuss weather for this goal, the naive is just the better option.

### v) Compute Your Own `E0s` (Required)

You need to compute your own `E0s` values for the level of theory you selected in `supporting/aims_command.py`. This affects the performance of fine-tuning significantly.

We provide an `atoms/` folder with an example script to do this.

What you need to do:
- Adapt the example script with your local AIMS paths.
- Set the same level of theory used in your `aims_command.py`. (very important! These are reference numbers for MACE.)
- Run the script and collect the resulting per-element reference energies.

Once done, replace the `heads -> default -> E0s` values in the `fine_tuning` block with your computed numbers.

## Step 2. Run Gen and Fine-Tune

In this step, we run the full Gen + fine-tuning loop and track the databases generated along the way.

High-level workflow:
- Gen produces candidate structures and stores them in generated databases.
- A subset is selected (FPS) and sent to labeling in batches.
- Labeling computes reference energies/forces with AIMS and writes labeled databases.
- MACE training runs on cumulative labeled data.
- The loop can continue until your target metrics are reached or you stop manually.

To run, just use the command:

`python3 -u $GENSEC_PATH/gensec/gensec.py parameters.json`

you can add this to your sbatch script. Replace `$GENSEC_PATH` with you gensec absolute path.

Main databases/files you will see:
- `db_generated_visual.db`: generated full structures (molecules on surface) for visualization/workflow transfer.
- `db_generated_fps.db`: FPS-selected subset used for fine-tuning input.
- `db_labeled_round_XXX.db`: labeled data for one loop round.
- `db_labeled_cumulative_XXX.db`: cumulative labeled training pool at a given round, used for training.
- `db_labeled_global.db`: persistent global labeled pool across rounds/runs.
- `db_labeled_test.db` (if test set is used): fixed labeled test set.
- `fine_tune_state.json`: loop state (where to continue, history, round index, etc.).

Restart options (state-file logic):
1. Continue from where you left off: keep `fine_tune_state.json` and rerun.
2. Re-do training in existing folders: delete the state file; useful if you changed fine-tuning/training parameters.
3. Fresh loop with existing labeled data: delete most run artifacts but keep `db_labeled_global.db` (or point to an external labeled DB). This can work, but treat it as unstable and validate carefully.

Final result of Step 2: **the model** (fine-tuned MACE model/checkpoints). TODO: **I need to make this better located, either move to main folder if converged or just keep copying the last model**.

## Step 3. New Calculation from Scratch with the Fine-Tuned Model

Now start a new calculation from scratch where this fine-tuned model is used as the main calculator.

What changes in `parameters.json` for this new run:
- Set the `calculator` block to your fine-tuned model calculator script.
- Remove (or disable) the `fine_tuning` block.
- Run the normal Gen -> Sec workflow with the improved model.